# Exploratory Data Analysis: Horse Market Prices (Avito.ma)

This notebook analyzes the scraped horse listings from Avito.ma to build a pricing engine.
**Goals:**
1. **Clean Prices**: Convert string prices (e.g., '10 000 DH') to numeric values.
2. **Extract Features**: Parse titles/descriptions for breed, age, and gender.
3. **Visualize Trends**: Price distribution by breed and location.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

# Config
DATA_PATH = Path("../backend/data/raw/avito_horses.json")
sns.set_theme(style="whitegrid")

## 1. Load & Inspect Data

In [ ]:
if not DATA_PATH.exists():
    print("Data file not found! Run the scraper first.")
else:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    df = pd.DataFrame(raw_data)
    print(f"Loaded {len(df)} listings")
    display(df.head())

## 2. Data Cleaning

**Tasks:**
- Remove 'DH' and spaces from Price.
- Handle 'Sur demande' or missing prices.
- Extract 'City' from Location.

In [ ]:
def clean_price(price_str):
    if not price_str or 'demande' in price_str.lower() or 'N/A' in price_str:
        return None
    # Remove non-numeric except potential decimals
    clean_str = re.sub(r'[^0-9]', '', price_str)
    try:
        return float(clean_str)
    except ValueError:
        return None

df['price_numeric'] = df['price'].apply(clean_price)

# Drop rows without price for the regression model, but keep for other analysis if needed
df_priced = df.dropna(subset=['price_numeric'])

print(f"Listings with valid price: {len(df_priced)} / {len(df)}")
df_priced['price_numeric'].describe()

## 3. Feature Extraction (Regex)

We need to extract **Age**, **Breed**, and **Gender** from the `title` text, as structured fields are missing.

In [ ]:
# Simple keywords for extraction (French)
BREEDS = ['arabe', 'barbe', 'anglo', 'espagnol', 'frison', 'poney', 'shetland', 'cheval']
GENDERS = {'jument': 'Female', 'poulinière': 'Female', 'hongre': 'Gelding', 'étalon': 'Male', 'poulain': 'Male', 'male': 'Male'}

def extract_breed(text):
    text = text.lower()
    found = []
    for b in BREEDS:
        if b in text:
            found.append(b)
    return found[0] if found else 'unknown'

def extract_gender(text):
    text = text.lower()
    for k, v in GENDERS.items():
        if k in text:
            return v
    return 'Unknown'

df_priced['extracted_breed'] = df_priced['title'].apply(extract_breed)
df_priced['extracted_gender'] = df_priced['title'].apply(extract_gender)

display(df_priced[['title', 'price_numeric', 'extracted_breed', 'extracted_gender']].head(10))

## 4. Visualizations

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_priced['price_numeric'], bins=30, kde=True)
plt.title('Price Distribution (MDH)')
plt.xlabel('Price')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='extracted_breed', y='price_numeric', data=df_priced)
plt.title('Price by Extracted Breed')
plt.xticks(rotation=45)
plt.show()